# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading, exploring, and analyzing the FAIR<sup>2</sup> dataset using the `mlcroissant` library. The dataset features mass spectrometry-based analyses of human extracellular vesicles and is described using the Croissant schema.

### Dataset Source
The dataset's Croissant schema is available at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from FAIR² using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset croissant metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review the available record sets, fields, and their Croissant `@id`s.

Below, we list the available RecordSets in the dataset, and display field `@id`s for each.

In [ ]:
# List RecordSet @id and their fields
print("Available RecordSets and fields in the dataset:")
recordset_list = []
recordsets = dataset.record_sets
for rs in recordsets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields and corresponding @ids:")
    for field in rs.fields:
        print(f"    - {field.name}: {field.id}")
    recordset_list.append(rs.id)
    print()

In [ ]:
# Show a sample record from each RecordSet
print("Example records from each RecordSet:")
for rs in dataset.record_sets:
    print(f"RecordSet: {rs.name} (@id: {rs.id})")
    recs = dataset.records(record_set=rs.id)
    try:
        example = next(recs)
        print(json.dumps(example, indent=2)[:500]+"...\n")
    except StopIteration:
        print("  No records found.\n")

## 3. Data Extraction
Extract data from RecordSets into pandas DataFrames for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Extract all record sets into DataFrames
record_sets = recordset_list # List of all RecordSet @ids
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded RecordSet {record_set_id}, shape: {df.shape}")
    except Exception as e:
        print(f"Could not load RecordSet {record_set_id}: {e}")

if len(dataframes) > 0:
    # Show column list and preview for the first record set
    first_recordset = list(dataframes.keys())[0]
    print(f"\nColumns for RecordSet {first_recordset}:\n{dataframes[first_recordset].columns.tolist()}")
    display(dataframes[first_recordset].head())

## 4. Exploratory Data Analysis (EDA)
Apply common EDA steps such as filtering, normalization, and grouping. We'll pick a numeric field for demonstration.

Modify the `numeric_field` and `group_field` variables according to available column `@id`s in your RecordSet.

In [ ]:
# Select the RecordSet and columns for EDA.
record_set_id = list(dataframes.keys())[0]
df = dataframes[record_set_id]

# Automatically try to pick a numeric field for demonstration
numeric_field = None
for col in df.columns:
    # try conversion, pick first successful
    try:
        vals = pd.to_numeric(df[col].dropna())
        if len(vals) > 0 and vals.dtype != 'O':
            numeric_field = col
            break
    except Exception:
        continue

print(f"Numeric field selected for analysis: {numeric_field}")
if numeric_field is None:
    raise RuntimeError("No numeric field found for EDA. Please inspect dataframes to select an appropriate field.")

# Convert column to numeric
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = df[numeric_field].quantile(0.75)  # Use upper quartile as threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold:.2f} (top 25%): {len(filtered_df)} rows")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} added:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field (pick a string type)
group_field = None
for col in df.columns:
    if col != numeric_field and df[col].dtype == 'O':
        unique_vals = df[col].nunique()
        if 1 < unique_vals < len(df) // 2:
            group_field = col
            break
print(f"Categorical field for grouping: {group_field}")
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean {numeric_field} by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and (optionally) the grouped means across a category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# If group_field is set, plot grouped means
if group_field and 'grouped_df' in locals():
    plt.figure(figsize=(10, 5))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f"Mean {numeric_field} grouped by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load, explore, and analyze the FAIR<sup>2</sup> protein mass spectrometry dataset according to its Croissant schema. Using the unique `@id` to reference RecordSets and fields, we listed dataset structure, loaded data to DataFrames, performed basic EDA, and visualized key metrics.

For deeper analysis, refer to the field documentation and scientific context attached to the dataset!